Link to the Logistic Regression model: https://drive.google.com/file/d/15N5L6AoOLRwKNYTZ0OLwiKphtISqqUe0/view?usp=drive_link

Link to the BERT model: https://drive.google.com/file/d/1z3B6FT1yDqhhbTsw__Wmy5q9Omt94ZBQ/view?usp=sharing

In [59]:
import spacy
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import classification_report, confusion_matrix
from spacy.tokens import Doc
import joblib
import json

In [60]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, Trainer
from datasets import Dataset
from collections import Counter

In [61]:
challenge_dataset = {
    "active_voice_arg0":[],   # Capability 1: ARG0 in active voice (MFT)
    "active_voice_arg1":[],   # Capability 1: ARG1 in active voice (MFT)
    "passive_voice_arg0":[],   # Capability 2: ARG0 in passive voice (MFT)
    "passive_voice_arg1":[],   # Capability 2: ARG1 in passive voice (MFT)
    "imperative":[],   # Capability 3: ARG1 in imperative sentences (MFT)
    "long_range_arg0":[],   # Capability 4: ARG0 in long-range dependency (MFT)
    "long_range_arg1":[],   # Capability 4: ARG1 in long-range dependency (MFT)
    "adjunct_ambiguity":[],   # Capability 5: PP attachment ARGM-MNR vs ARGM-COM (MFT)
}

# Capability 1: ARG0 and ARG1 in Active Voice (MFT)
# Tests basic agent and patient detection in active transitive sentences.

#arg0
active_voice_instances = [
    {
        "tokens": ["The", "chef", "cooked", "the", "fish", "."],
        "predicate_mask": [0, 0, 1, 0, 0, 0],
        "expected": {1: "ARG0"},
        "test": "active_voice_arg0",
        "description": "ARG0 detection: basic active"
    },
    {
        "tokens": ["The", "dog", "bit", "the", "man", "."],
        "predicate_mask": [0, 0, 1, 0, 0, 0],
        "expected": {1: "ARG0"},
        "test": "active_voice_arg0",
        "description": "ARG0 detection: animal agent"
    },
    {
        "tokens": ["Maria", "wrote", "a", "letter", "."],
        "predicate_mask": [0, 1, 0, 0, 0],
        "expected": {0: "ARG0"},
        "test": "active_voice_arg0",
        "description": "ARG0 detection: proper noun agent"
    },
    {
        "tokens": ["The", "student", "read", "the", "book", "."],
        "predicate_mask": [0, 0, 1, 0, 0, 0],
        "expected": {1: "ARG0"},
        "test": "active_voice_arg0",
        "description": "ARG0 detection: cognitive verb"
    },
    {
        "tokens": ["John", "kicked", "the", "ball", "."],
        "predicate_mask": [0, 1, 0, 0, 0],
        "expected": {0: "ARG0"},
        "test": "active_voice_arg0",
        "description": "ARG0 detection: proper noun physical action"
    },
    {
        "tokens": ["The", "company", "hired", "a", "new", "employee", "."],
        "predicate_mask": [0, 0, 1, 0, 0, 0, 0],
        "expected": {1: "ARG0"},
        "test": "active_voice_arg0",
        "description": "ARG0 detection: organization as agent"
    },
    {
        "tokens": ["Scientists", "discovered", "a", "new", "planet", "."],
        "predicate_mask": [0, 1, 0, 0, 0, 0],
        "expected": {0: "ARG0"},
        "test": "active_voice_arg0",
        "description": "ARG0 detection: plural agent"
    },
    {
        "tokens": ["He", "broke", "the", "window", "."],
        "predicate_mask": [0, 1, 0, 0, 0],
        "expected": {0: "ARG0"},
        "test": "active_voice_arg0",
        "description": "ARG0 detection: pronoun as agent"
    },
    #arg1
    {
        "tokens": ["The", "chef", "cooked", "the", "fish", "."],
        "predicate_mask": [0, 0, 1, 0, 0, 0],
        "expected": {4: "ARG1"},
        "test": "active_voice_arg1",
        "description": "ARG1 detection: basic active"
    },
    {
        "tokens": ["The", "dog", "bit", "the", "man", "."],
        "predicate_mask": [0, 0, 1, 0, 0, 0],
        "expected": {4: "ARG1"},
        "test": "active_voice_arg1",
        "description": "ARG1 detection: animate patient"
    },
    {
        "tokens": ["Maria", "wrote", "a", "letter", "."],
        "predicate_mask": [0, 1, 0, 0, 0],
        "expected": {3: "ARG1"},
        "test": "active_voice_arg1",
        "description": "ARG1 detection: indefinite NP patient"
    },
    {
        "tokens": ["The", "student", "read", "the", "book", "."],
        "predicate_mask": [0, 0, 1, 0, 0, 0],
        "expected": {4: "ARG1"},
        "test": "active_voice_arg1",
        "description": "ARG1 detection: cognitive verb"
    },
    {
        "tokens": ["John", "kicked", "the", "ball", "."],
        "predicate_mask": [0, 1, 0, 0, 0],
        "expected": {3: "ARG1"},
        "test": "active_voice_arg1",
        "description": "ARG1 detection: inanimate object"
    },
    {
        "tokens": ["The", "company", "hired", "a", "new", "employee", "."],
        "predicate_mask": [0, 0, 1, 0, 0, 0, 0],
        "expected": {5: "ARG1"},
        "test": "active_voice_arg1",
        "description": "ARG1 detection: patient with modifier"
    },
    {
        "tokens": ["Scientists", "discovered", "a", "new", "planet", "."],
        "predicate_mask": [0, 1, 0, 0, 0, 0],
        "expected": {4: "ARG1"},
        "test": "active_voice_arg1",
        "description": "ARG1 detection: inanimate patient with modifier"
    },
    {
        "tokens": ["He", "broke", "the", "window", "."],
        "predicate_mask": [0, 1, 0, 0, 0],
        "expected": {3: "ARG1"},
        "test": "active_voice_arg1",
        "description": "ARG1 detection: inanimate patient with pronoun agent"
    },
]

challenge_dataset["active_voice_arg0"] = [i for i in active_voice_instances if i["test"] == "active_voice_arg0"]
challenge_dataset["active_voice_arg1"] = [i for i in active_voice_instances if i["test"] == "active_voice_arg1"]

# Capability 2: ARG0 and ARG1 in Passive Voice (MFT)
# Tests agent and patient detection in passive voice (patient is subject and agent moves to by-phrase or disappears)

passive_voice_instances = [
    # ARG1 in passive 
    {
        "tokens": ["The", "fish", "was", "cooked", "by", "the", "chef", "."],
        "predicate_mask": [0, 0, 0, 1, 0, 0, 0, 0],
        "expected": {1: "ARG1"},
        "test": "passive_voice_arg1",
        "description": "ARG1 detection: patient as subject"
    },
    {
        "tokens": ["The", "man", "was", "bitten", "by", "the", "dog", "."],
        "predicate_mask": [0, 0, 0, 1, 0, 0, 0, 0],
        "expected": {1: "ARG1"},
        "test": "passive_voice_arg1",
        "description": "ARG1 detection: animate patient as subject"
    },
    {
        "tokens": ["A", "letter", "was", "written", "by", "Maria", "."],
        "predicate_mask": [0, 0, 0, 1, 0, 0, 0],
        "expected": {1: "ARG1"},
        "test": "passive_voice_arg1",
        "description": "ARG1 detection: indefinite NP as subject"
    },
    {
        "tokens": ["The", "book", "was", "read", "by", "the", "student", "."],
        "predicate_mask": [0, 0, 0, 1, 0, 0, 0, 0],
        "expected": {1: "ARG1"},
        "test": "passive_voice_arg1",
        "description": "ARG1 detection: cognitive verb passive"
    },
    {
        "tokens": ["The", "ball", "was", "kicked", "by", "John", "."],
        "predicate_mask": [0, 0, 0, 1, 0, 0, 0],
        "expected": {1: "ARG1"},
        "test": "passive_voice_arg1",
        "description": "ARG1 detection: inanimate patient as subject"
    },
    {
        "tokens": ["The", "window", "was", "broken", "."],
        "predicate_mask": [0, 0, 0, 1, 0],
        "expected": {1: "ARG1"},
        "test": "passive_voice_arg1",
        "description": "ARG1 detection: agentless passive"
    },
    {
        "tokens": ["The", "letter", "was", "delivered", "."],
        "predicate_mask": [0, 0, 0, 1, 0],
        "expected": {1: "ARG1"},
        "test": "passive_voice_arg1",
        "description": "ARG1 detection: agentless passive"
    },
    {
        "tokens": ["Mistakes", "were", "made", "."],
        "predicate_mask": [0, 0, 1, 0],
        "expected": {0: "ARG1"},
        "test": "passive_voice_arg1",
        "description": "ARG1 detection: plural subject agentless passive"
    },
    # ARG0 in passive (agent in by-phrase)
    {
        "tokens": ["The", "fish", "was", "cooked", "by", "the", "chef", "."],
        "predicate_mask": [0, 0, 0, 1, 0, 0, 0, 0],
        "expected": {6: "ARG0"},
        "test": "passive_voice_arg0",
        "description": "ARG0 detection: agent in by-phrase"
    },
    {
        "tokens": ["The", "man", "was", "bitten", "by", "the", "dog", "."],
        "predicate_mask": [0, 0, 0, 1, 0, 0, 0, 0],
        "expected": {6: "ARG0"},
        "test": "passive_voice_arg0",
        "description": "ARG0 detection: animal agent in by-phrase"
    },
    {
        "tokens": ["A", "letter", "was", "written", "by", "Maria", "."],
        "predicate_mask": [0, 0, 0, 1, 0, 0, 0],
        "expected": {5: "ARG0"},
        "test": "passive_voice_arg0",
        "description": "ARG0 detection: proper noun in by-phrase"
    },
    {
        "tokens": ["The", "book", "was", "read", "by", "the", "student", "."],
        "predicate_mask": [0, 0, 0, 1, 0, 0, 0, 0],
        "expected": {6: "ARG0"},
        "test": "passive_voice_arg0",
        "description": "ARG0 detection: cognitive verb agent in by-phrase"
    },
    {
        "tokens": ["The", "ball", "was", "kicked", "by", "John", "."],
        "predicate_mask": [0, 0, 0, 1, 0, 0, 0],
        "expected": {5: "ARG0"},
        "test": "passive_voice_arg0",
        "description": "ARG0 detection: proper noun agent in by-phrase"
    },
    {
        "tokens": ["The", "report", "was", "written", "by", "the", "team", "."],
        "predicate_mask": [0, 0, 0, 1, 0, 0, 0, 0],
        "expected": {6: "ARG0"},
        "test": "passive_voice_arg0",
        "description": "ARG0 detection: group agent in by-phrase"
    },
    {
        "tokens": ["The", "prize", "was", "won", "by", "the", "student", "."],
        "predicate_mask": [0, 0, 0, 1, 0, 0, 0, 0],
        "expected": {6: "ARG0"},
        "test": "passive_voice_arg0",
        "description": "ARG0 detection: achievement verb agent in by-phrase"
    },
    {
        "tokens": ["The", "song", "was", "sung", "by", "the", "choir", "."],
        "predicate_mask": [0, 0, 0, 1, 0, 0, 0, 0],
        "expected": {6: "ARG0"},
        "test": "passive_voice_arg0",
        "description": "ARG0 detection: group agent performance verb"
    },
]

challenge_dataset["passive_voice_arg1"] = [i for i in passive_voice_instances if i["test"] == "passive_voice_arg1"]
challenge_dataset["passive_voice_arg0"] = [i for i in passive_voice_instances if i["test"] == "passive_voice_arg0"]

# Capability 3: Imperative Sentences (MFT)
# Tests whether ARG1 is correctly identified when ARG0 is implicit.

imperative_instances = [
    {
        "tokens": ["Eat", "the", "apple", "."],
        "predicate_mask": [1, 0, 0, 0],
        "expected": {2: "ARG1"},
        "test": "imperative",
        "description": "Basic imperative: ARG1 with no explicit ARG0"
    },
    {
        "tokens": ["Close", "the", "door", "."],
        "predicate_mask": [1, 0, 0, 0],
        "expected": {2: "ARG1"},
        "test": "imperative",
        "description": "Basic imperative: physical action"
    },
    {
        "tokens": ["Read", "the", "instructions", "carefully", "."],
        "predicate_mask": [1, 0, 0, 0, 0],
        "expected": {2: "ARG1"},
        "test": "imperative",
        "description": "Imperative with adverb"
    },
    {
        "tokens": ["Call", "your", "mother", "."],
        "predicate_mask": [1, 0, 0, 0],
        "expected": {2: "ARG1"},
        "test": "imperative",
        "description": "Imperative with possessive in ARG1"
    },
    {
        "tokens": ["Open", "the", "window", "."],
        "predicate_mask": [1, 0, 0, 0],
        "expected": {2: "ARG1"},
        "test": "imperative",
        "description": "Basic imperative: physical action"
    },
    {
        "tokens": ["Write", "your", "name", "on", "the", "form", "."],
        "predicate_mask": [1, 0, 0, 0, 0, 0, 0],
        "expected": {2: "ARG1"},
        "test": "imperative",
        "description": "Imperative with locative PP"
    },
    {
        "tokens": ["Finish", "your", "homework", "."],
        "predicate_mask": [1, 0, 0, 0],
        "expected": {2: "ARG1"},
        "test": "imperative",
        "description": "Basic imperative: task-oriented"
    },
    {
        "tokens": ["Pass", "the", "salt", "."],
        "predicate_mask": [1, 0, 0, 0],
        "expected": {2: "ARG1"},
        "test": "imperative",
        "description": "Basic imperative: request"
    },
]

challenge_dataset["imperative"] = imperative_instances

# Capability 4: Long-Range Dependency (MFT)
#Tests ARG0 and ARG1 detection when there are many tokens between the argument and the predicate (relative clauses, embedded clauses, long NPs)
long_range_instances = [
    # ARG0 with relative clause between subject and predicate
    {
        "tokens": ["The", "man", "who", "bought", "the", "car", "drove", "the", "vehicle", "home", "."],
        "predicate_mask": [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
        "expected": {1: "ARG0"},
        "test": "long_range_arg0",
        "description": "ARG0 detection: subject with relative clause"
    },
    {
        "tokens": ["The", "scientist", "who", "won", "the", "prize", "published", "a", "paper", "."],
        "predicate_mask": [0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
        "expected": {1: "ARG0"},
        "test": "long_range_arg0",
        "description": "ARG0 detection: subject with relative clause"
    },
    {
        "tokens": ["The", "woman", "that", "everyone", "admired", "received", "an", "award", "."],
        "predicate_mask": [0, 0, 0, 0, 0, 1, 0, 0, 0],
        "expected": {1: "ARG0"},
        "test": "long_range_arg0",
        "description": "ARG0 detection: subject with that-clause"
    },
    {
        "tokens": ["The", "company", "that", "built", "the", "bridge", "went", "bankrupt", "."],
        "predicate_mask": [0, 0, 0, 0, 0, 0, 1, 0, 0],
        "expected": {1: "ARG0"},
        "test": "long_range_arg0",
        "description": "ARG0 detection: organization subject with relative clause"
    },
    {
        "tokens": ["The", "student", "who", "studied", "all", "night", "passed", "the", "exam", "."],
        "predicate_mask": [0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
        "expected": {1: "ARG0"},
        "test": "long_range_arg0",
        "description": "ARG0 detection: subject with adverbial relative clause"
    },
    {
        "tokens": ["The", "dog", "that", "chased", "the", "cat", "barked", "loudly", "."],
        "predicate_mask": [0, 0, 0, 0, 0, 0, 1, 0, 0],
        "expected": {1: "ARG0"},
        "test": "long_range_arg0",
        "description": "ARG0 detection: animal subject with relative clause"
    },
    {
        "tokens": ["The", "lawyer", "who", "defended", "the", "suspect", "won", "the", "case", "."],
        "predicate_mask": [0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
        "expected": {1: "ARG0"},
        "test": "long_range_arg0",
        "description": "ARG0 detection: occupation subject with relative clause"
    },
    {
        "tokens": ["The", "children", "who", "played", "in", "the", "park", "went", "home", "."],
        "predicate_mask": [0, 0, 0, 0, 0, 0, 0, 1, 0, 0],
        "expected": {1: "ARG0"},
        "test": "long_range_arg0",
        "description": "ARG0 detection: plural subject with relative clause"
    },
    # ARG1 with embedded clause between predicate and object
    {
        "tokens": ["The", "man", "who", "bought", "the", "car", "drove", "the", "vehicle", "home", "."],
        "predicate_mask": [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
        "expected": {8: "ARG1"},
        "test": "long_range_arg1",
        "description": "ARG1 detection: object after long subject"
    },
    {
        "tokens": ["The", "scientist", "who", "won", "the", "prize", "published", "a", "paper", "."],
        "predicate_mask": [0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
        "expected": {8: "ARG1"},
        "test": "long_range_arg1",
        "description": "ARG1 detection: object after subject with relative clause"
    },
    {
        "tokens": ["The", "woman", "that", "everyone", "admired", "received", "an", "award", "."],
        "predicate_mask": [0, 0, 0, 0, 0, 1, 0, 0, 0],
        "expected": {7: "ARG1"},
        "test": "long_range_arg1",
        "description": "ARG1 detection: object after that-clause subject"
    },
    {
        "tokens": ["The", "student", "who", "studied", "all", "night", "passed", "the", "exam", "."],
        "predicate_mask": [0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
        "expected": {8: "ARG1"},
        "test": "long_range_arg1",
        "description": "ARG1 detection: object after subject with adverbial clause"
    },
    {
        "tokens": ["The", "lawyer", "who", "defended", "the", "suspect", "won", "the", "case", "."],
        "predicate_mask": [0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
        "expected": {8: "ARG1"},
        "test": "long_range_arg1",
        "description": "ARG1 detection: object after relative clause subject"
    },
    {
        "tokens": ["The", "company", "that", "built", "the", "bridge", "announced", "new", "plans", "."],
        "predicate_mask": [0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
        "expected": {8: "ARG1"},
        "test": "long_range_arg1",
        "description": "ARG1 detection: object after organization subject with relative clause"
    },
    {
        "tokens": ["The", "team", "that", "lost", "the", "match", "left", "the", "stadium", "."],
        "predicate_mask": [0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
        "expected": {8: "ARG1"},
        "test": "long_range_arg1",
        "description": "ARG1 detection: object after team subject with relative clause"
    },
    {
        "tokens": ["The", "boy", "who", "found", "the", "wallet", "returned", "it", "."],
        "predicate_mask": [0, 0, 0, 0, 0, 0, 1, 0],
        "expected": {7: "ARG1"},
        "test": "long_range_arg1",
        "description": "ARG1 detection: pronoun object after relative clause subject"
    },
]

challenge_dataset["long_range_arg0"] = [i for i in long_range_instances if i["test"] == "long_range_arg0"]
challenge_dataset["long_range_arg1"] = [i for i in long_range_instances if i["test"] == "long_range_arg1"]

# Capability 5: Adjunct role dismbiguation (MFT)
# Tests distinguishing btw instrument PPs (ARGM-MNR) from comitative PPs (ARGM-COM) when both use the preposition "with".
adjunct_ambiguity_instances = [
    {
        "tokens": ["I", "ate", "the", "soup", "with", "a", "spoon", "."],
        "predicate_mask": [0, 1, 0, 0, 0, 0, 0, 0],
        "expected": {6: "ARGM-MNR"},
        "test": "adjunct_ambiguity",
        "pp_type": "instrument",
        "description": "Instrument PP"
    },
    {
        "tokens": ["I", "ate", "the", "soup", "with", "my", "sister", "."],
        "predicate_mask": [0, 1, 0, 0, 0, 0, 0, 0],
        "expected": {6: "ARGM-COM"},
        "test": "adjunct_ambiguity",
        "pp_type": "comitative",
        "description": "Comitative PP"
    },
    {
        "tokens": ["He", "wrote", "the", "letter", "with", "a", "pen", "."],
        "predicate_mask": [0, 1, 0, 0, 0, 0, 0, 0],
        "expected": {6: "ARGM-MNR"},
        "test": "adjunct_ambiguity",
        "pp_type": "instrument",
        "description": "Instrument PP"
    },
    {
        "tokens": ["He", "wrote", "the", "letter", "with", "his", "friend", "."],
        "predicate_mask": [0, 1, 0, 0, 0, 0, 0, 0],
        "expected": {6: "ARGM-COM"},
        "test": "adjunct_ambiguity",
        "pp_type": "comitative",
        "description": "Comitative PP"
    },
    {
        "tokens": ["She", "cut", "the", "bread", "with", "a", "knife", "."],
        "predicate_mask": [0, 1, 0, 0, 0, 0, 0, 0],
        "expected": {6: "ARGM-MNR"},
        "test": "adjunct_ambiguity",
        "pp_type": "instrument",
        "description": "Instrument PP"
    },
    {
        "tokens": ["She", "cut", "the", "bread", "with", "her", "mother", "."],
        "predicate_mask": [0, 1, 0, 0, 0, 0, 0, 0],
        "expected": {6: "ARGM-COM"},
        "test": "adjunct_ambiguity",
        "pp_type": "comitative",
        "description": "Comitative PP"
    },
    {
        "tokens": ["The", "boy", "hit", "the", "ball", "with", "a", "bat", "."],
        "predicate_mask": [0, 0, 1, 0, 0, 0, 0, 0, 0],
        "expected": {7: "ARGM-MNR"},
        "test": "adjunct_ambiguity",
        "pp_type": "instrument",
        "description": "Instrument PP"
    },
    {
        "tokens": ["The", "boy", "hit", "the", "ball", "with", "his", "teammate", "."],
        "predicate_mask": [0, 0, 1, 0, 0, 0, 0, 0, 0],
        "expected": {7: "ARGM-COM"},
        "test": "adjunct_ambiguity",
        "pp_type": "comitative",
        "description": "Comitative PP"
    },
]

challenge_dataset["adjunct_ambiguity"] = adjunct_ambiguity_instances

# Save dataset
with open('challenge_dataset.json', 'w', encoding='utf-8') as f:
    json.dump(challenge_dataset, f, indent=2)

print("Challenge dataset saved.")
for test_name, instances in challenge_dataset.items():
    print(f"  {test_name}: {len(instances)} instances")

Challenge dataset saved.
  active_voice_arg0: 8 instances
  active_voice_arg1: 8 instances
  passive_voice_arg0: 8 instances
  passive_voice_arg1: 8 instances
  imperative: 8 instances
  long_range_arg0: 8 instances
  long_range_arg1: 8 instances
  adjunct_ambiguity: 8 instances


In [62]:
import joblib
import spacy
from spacy.tokens import Doc
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

# A1 Logistic Regression
a1_bundle   = joblib.load('./srl_model.joblib')
a1_model    = a1_bundle['model']
a1_vec      = a1_bundle['vectorizer']
nlp         = spacy.load("en_core_web_md")

#A2:BERT
from transformers import AutoTokenizer, AutoModelForTokenClassification, Trainer
from datasets import Dataset
import numpy as np

from transformers import AutoConfig
A2_MODEL_PATH = './srl_model_seed_42'
config = AutoConfig.from_pretrained(A2_MODEL_PATH)
id2label = config.id2label

In [63]:
nlp = spacy.load("en_core_web_md")

def get_spacy_parse(tokens):
    """
    Parses a list of tokens using spaCy without retokenizing.
    Args:
        tokens (list[str]): A list of pre-tokenized words representing a sentence.

    Returns:
        list[list[str]]: A list of token rows in a CoNLL-style format.
        Each row has 11 columns, and only some are filled:
            row[1] : token text
            row[2] : lemma
            row[3] : POS tag
            row[6] : syntactic head index (1-based)
            row[7] : dependency relation label
        Other columns are empty placeholders.
    """
    #create Doc directly from tokens(spaCy skips its tokenizer)
    doc = Doc(nlp.vocab, words=tokens)
    # Run the spaCy pipeline
    doc = nlp(doc)
    
    parsed_sent = []
    for token in doc:
        # Create an empty row with 11 columns to match CoNLL structure
        row = [''] * 11
        row[1] = token.text  #word form
        row[2] = token.lemma_ #lemma
        row[3] = token.pos_  #POS tag
        row[6] = str(token.head.i + 1)  # # head index (1-indexed to match CoNLL)
        row[7] = token.dep_  #dep rel label
        parsed_sent.append(row)
    
    return parsed_sent

In [64]:
def get_dep_path_to_predicate(sent, token_idx, predicate_idx):
    """
    Get the dependency path from a token to a predicate, 
    by going upward from the token to the Lowest Common Ancestor (LCA), and downward from the LCA to the predicate.

    Directions are encoded as:
        ↑ : moving up the dependency tree (toward the root)
        ↓ : moving down the dependency tree (toward the predicate)
        
    The predicate lemma is appended to the path to use predicate information.
    
    Args:
        sent (list[list[str]]): A sentence in CoNLL-like format.
        token_idx (int): Index of the token.
        predicate_idx (int): Index of the predicate token.

    Returns: str: Dependency path in the form of "dep↑>dep↑>dep↓>dep↓|predicate_lemma"
    """
    # Get the predicate lemma
    predicate_lemma = sent[predicate_idx][2]
     # If the token itself is the predicate
    if token_idx == predicate_idx:
        return f"SELF|{predicate_lemma}"
    
    head   = {}
    deprel = {}
    # Build head and dependency relation dictionaries
    for i, tok in enumerate(sent):
        # safely parse head index 
        try:
            head[i] = int(tok[6]) - 1
        except (ValueError, IndexError):
            head[i] = -1  # treat missing value as root
        deprel[i] = tok[7] if len(tok) > 7 else 'UNK' # Get dependency relation
    
    def get_ancestors(idx):
        """
        Go upward in the dependency tree and collect ancestors.
        Returns a list of (node_index, dependency_relation) pairs.
        """
        path = []
        visited = set()
        current = idx
        while current != -1:
            if current in visited:
                break
            visited.add(current)
            path.append((current, deprel.get(current, '')))
            current = head.get(current, -1)
        return path
    # Get ancestor paths for token and predicate
    token_ancestors = get_ancestors(token_idx)
    pred_ancestors = get_ancestors(predicate_idx)
    # Set of predicate ancestor nodes
    pred_ancestor_nodes = {node for node, dep in pred_ancestors}
    # Find Lowest Common Ancestor (LCA)
    lca = None
    token_path_up = []
    for node, dep in token_ancestors:
        if node in pred_ancestor_nodes:
            lca = node
            break
        token_path_up.append(dep + "↑")
    # If no common ancestor exists
    if lca is None:
        return f"NOPATH|{predicate_lemma}"
    # Build path from predicate up to LCA
    pred_path_up = []
    for node, dep in pred_ancestors:
        if node == lca:
            break
        pred_path_up.append(dep + "↓")
    pred_path_down = list(reversed(pred_path_up)) #Reverse to get LCA → predicate direction
    # Combine upward and downward paths
    full_path = token_path_up + pred_path_down
    # Convert to string 
    path_str = ">".join(full_path) if full_path else "SELF"
    
    return f"{path_str}|{predicate_lemma}"

In [65]:
def extract_features(instance):
    """
    Extract features for a SRL instance, including:
        - dependency path from the token to the predicate+predicate's lemma
        - token lemma
        - token POS tag
    Args:
        instance (dict): An instance produced by `preprocess`
    Returns: list[dict]
    """
    tokens = instance['tokens']
    pred_idx = instance['predicate_idx']
    # Parse the sentence with spaCy
    parsed_sent = get_spacy_parse(tokens)  
    # Extract features for each token
    features = []
    for i in range(len(tokens)):
        feat = {
            'dep_path':   get_dep_path_to_predicate(parsed_sent, i, pred_idx),
            'token_lemma': parsed_sent[i][2],   #lemma from spaCy
            'token_pos':   parsed_sent[i][3],   #POS from spaCy
        }
        features.append(feat)
    
    return features

In [66]:
def predict_srl_a1(sentence_tokens, predicate_mask, model, vectorizer):
    """
    Performs SRL on a new sentence given the predicate position.
    
    Args:
        sentence_tokens: list of strings e.g. ['Pia', 'asked', 'Luis', '.']
        predicate_mask:  list of 0/1    e.g. [0, 1, 0, 0]  (1 = predicate)
        model:           trained LogisticRegression
        vectorizer:      fitted DictVectorizer
    
    Returns:
        list of predicted SRL labels.
    """
    #Get predicate index from the mask
    predicate_idx = predicate_mask.index(1)
    # Create an instance in the same format as training 
    instance = {
        'tokens':        sentence_tokens,
        'predicate_idx': predicate_idx,
    }
    # Extract feature dictionaries using spaCy
    features = extract_features(instance)  
    X = vectorizer.transform(features)
    preds = model.predict(X)
    return list(preds)


In [67]:
def mark_predicate(tokens, predicate_idx):  #making the predicate explicit in the input
    """
    Marks the predicate token by wrapping it with [PRED] markers.
    """
    marked = tokens.copy()
    marked[predicate_idx] = f"[PRED]{tokens[predicate_idx]}[/PRED]"
    return marked

In [68]:
def align_predictions_with_tokens(predictions, original_tokens_list, loaded_tokenizer=None):
    """
    Converts subword-level predictions back to token-level predictions.
    Uses majority voting among subword predictions, and first subword prediction in case of a tie.
    """
    tok = loaded_tokenizer if loaded_tokenizer is not None else tokenizer
    token_predictions = []

    for pred_seq, orig_tokens in zip(predictions, original_tokens_list):
        # Tokenize to get word_ids mapping
        tokenized = tok(orig_tokens, is_split_into_words=True, truncation=True)
        word_ids  = tokenized.word_ids()

        # Store votes and first label for each original token
        votes        = {}
        first_labels = {}
        
        for i, word_idx in enumerate(word_ids):
            # Skip special tokens
            if word_idx is None:
                continue
            # Get label prediction for current subword
            label_id = int(pred_seq[i])
            # Group predictions for subwords with same word_idx
            if word_idx not in votes:
                votes[word_idx]        = []
                first_labels[word_idx] = label_id
            votes[word_idx].append(label_id)
            
        #majority vote for each token
        sentence_preds = []
        for word_idx in sorted(votes.keys()):
            word_votes = votes[word_idx]
            counts     = Counter(word_votes)
            max_freq   = max(counts.values()) #max frequency
            candidates = [label for label, count in counts.items() if count == max_freq] #Collect all candidates with max frequency
            
            #use first subword prediction in case of a tie
            final_label = first_labels[word_idx] if len(candidates) > 1 else candidates[0]
            sentence_preds.append(id2label[final_label])  #convert int to SRL label string
        
        token_predictions.append(sentence_preds)
    
    return token_predictions

In [69]:
def predict_srl_a2(sentence_tokens, predicate_mask, model_path='./srl_model_seed_42'):
    """
    Performs SRL on a new sentence given the predicate position.
    
    Args:
        sentence_tokens: list of strings e.g. ['Pia', 'asked', 'Luis', '.']
        predicate_mask:  list of 0/1    e.g. [0, 1, 0, 0]  (1 = predicate)
        model_path:      path to saved model
    
    Returns:
        list of predicted SRL labels, one per token
    """
    predicate_idx = predicate_mask.index(1)
    
    # mark predicate 
    marked_tokens = mark_predicate(sentence_tokens, predicate_idx)
    
    # load model and tokenizer
    loaded_model     = AutoModelForTokenClassification.from_pretrained(model_path)
    loaded_tokenizer = AutoTokenizer.from_pretrained(model_path)
    
    # tokenize
    tokenized = loaded_tokenizer(
        marked_tokens,
        is_split_into_words=True,
        truncation=True,
        return_tensors="pt"
    )
    
    # predict
    loaded_trainer = Trainer(model=loaded_model)
    standalone_ds  = Dataset.from_list([{k: v[0].tolist() for k, v in tokenized.items()}])
    preds, _, _    = loaded_trainer.predict(standalone_ds)
    preds          = np.argmax(preds, axis=2)
    
    # align subwords back to tokens
    aligned = align_predictions_with_tokens(preds, [marked_tokens], loaded_tokenizer)
    return aligned[0]


In [71]:
def run_model_on_dataset(challenge_dataset, predict_fn):
    """
    Runs a predict function on all instances in the challenge dataset.
    Returns a dict of {test_name: list of prediction lists}
    """
    results = {}
    
    for test_name, instances in challenge_dataset.items():
        results[test_name] = []
        for inst in instances:
            preds = predict_fn(inst['tokens'], inst['predicate_mask'])
            results[test_name].append(preds)
    
    return results


def run_a1(tokens, mask):
    return predict_srl_a1(tokens, mask, a1_model, a1_vec)

def run_a2(tokens, mask):
    return predict_srl_a2(tokens, mask)

a1_results = run_model_on_dataset(challenge_dataset, run_a1)
a2_results = run_model_on_dataset(challenge_dataset, run_a2)

In [72]:
def compute_failure_rate(challenge_dataset, model_results):
    """
    Computes failure rate per test (number of instances where any expected label is wrong/ total instances in test)
    """
    failure_rates = {}
    
    for test_name, instances in challenge_dataset.items():
        preds = model_results[test_name]
        n_failed = 0
        
        for inst, pred in zip(instances, preds):
            failed = False
            for token_idx, expected_label in inst['expected'].items():
                predicted_label = pred[token_idx]
                if predicted_label != expected_label:
                    failed = True
                    break  #one wrong=instance fails
            if failed:
                n_failed += 1
        
        failure_rates[test_name] = {
            'failed':   n_failed,
            'total':    len(instances),
            'failure_rate': n_failed / len(instances) if instances else 0
        }
    
    return failure_rates


a1_failure_rates = compute_failure_rate(challenge_dataset, a1_results)
a2_failure_rates = compute_failure_rate(challenge_dataset, a2_results)

print("Failure Rates")
print(f"{'Test':<25} {'A1 (LR)':<15} {'A2 (BERT)':<15}")
print("-" * 55)
for test_name in challenge_dataset:
    a1_fr = a1_failure_rates[test_name]['failure_rate']
    a2_fr = a2_failure_rates[test_name]['failure_rate']
    print(f"{test_name:<25} {a1_fr:.2f} ({a1_failure_rates[test_name]['failed']}/{a1_failure_rates[test_name]['total']})    {a2_fr:.2f} ({a2_failure_rates[test_name]['failed']}/{a2_failure_rates[test_name]['total']})")

Failure Rates
Test                      A1 (LR)         A2 (BERT)      
-------------------------------------------------------
active_voice_arg0         0.62 (5/8)    0.00 (0/8)
active_voice_arg1         0.38 (3/8)    0.00 (0/8)
passive_voice_arg0        1.00 (8/8)    0.00 (0/8)
passive_voice_arg1        0.62 (5/8)    0.12 (1/8)
imperative                0.00 (0/8)    0.00 (0/8)
long_range_arg0           0.25 (2/8)    0.12 (1/8)
long_range_arg1           0.38 (3/8)    0.00 (0/8)
adjunct_ambiguity         1.00 (8/8)    0.50 (4/8)


In [73]:
def save_model_output(challenge_dataset, model_results, model_name, filepath):
    """
    Saves model predictions on challenge dataset in json format.
    """
    output = {}
    
    for test_name, instances in challenge_dataset.items():
        output[test_name] = []
        preds = model_results[test_name]
        
        for inst, pred in zip(instances, preds):
            output[test_name].append({
                'tokens':      inst['tokens'],
                'predicate_mask': inst['predicate_mask'],
                'expected':    inst['expected'],
                'predicted':   {str(k): pred[k] for k in inst['expected']},
                'description': inst['description'],
                'passed':      all(pred[k] == v for k, v in inst['expected'].items())
            })
    
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(output, f, indent=2)
    
    print(f"Saved {model_name} output to {filepath}")


save_model_output(challenge_dataset, a1_results, 'A1_LogReg', 'a1_challenge_output.json')
save_model_output(challenge_dataset, a2_results, 'A2_BERT',   'a2_challenge_output.json')

Saved A1_LogReg output to a1_challenge_output.json
Saved A2_BERT output to a2_challenge_output.json
